[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/cga_spherical_quadrics/cga_spherical_quadrics.ipynb)

# Spherical Quadrics in a Conformal Flow

This example draws curved regions on a sphere, donuts, figure eights and hourglasses, and carries them all around the sphere by one smooth conformal motion, a flow that keeps angles but stretches and shrinks lengths as it goes. Each region is a quadric of the sphere: the points where a quadratic expression is negative. The flow turns around two fixed points where two circles cross, and after a full turn every region is back where it started.

In the conformal algebra of the sphere, `Cl(3, 1)` with `w` squaring to minus one, a circle on the sphere is a vector, `n + s * mv.w` for the plane `n | x == s` that cuts it out, and a point of the sphere is the dual of its null vector. A circle paired with a point, `circle & point`, is positive inside the circle. A quadric is a map from points to planes, `Plane <- Point`, a sum of circle dyads such as `outer * (inner & Point)`, and a point lies inside where `quadric(point) & point < 0`. Two circles meet in the 2-blade of their wedge, whose exponential is a conformal rotor, and a rotor moves a quadric as it moves any map, `versor >> quadric(versor << Point)`.

In the notation of conformal geometry the quadrics read as symmetric $4 \times 4$ matrices on the homogeneous coordinates of the sphere, the conformal motions as the Möbius transformations, the group $SO^+(3, 1)$ acting on them by congruence, and the flow as a one-parameter subgroup, loxodromic in general.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Extensor, stack
from examples.animation import show_animation
from examples.quadrics.cga_spherical_quadrics import render
from examples.quadrics.cga_spherical_quadrics.core import Bivector, Plane, Point, Quadric, Rotor, mv

np.set_printoptions(precision=4, suppress=True)

## 1. Points and moving maps

A point of the sphere at unit coordinates is the dual of its null vector, its coordinates on `x`, `y` and `z` plus `w`. A quadric moves by a versor as any map does: the point pulled back, the plane pushed forward.

In [ ]:
def point(xyz: np.ndarray) -> Point:
    """The points of S² at (..., 3) unit coordinates: the duals of their null vectors, x + y + z + w."""
    return ((stack((mv.x, mv.y, mv.z)) * xyz).sum(axis=-1) + mv.w).dual()


def moved(quadric: Quadric, versor: Rotor) -> Quadric:
    """A quadric carried by a versor: pull the point back, push the plane forward."""
    return versor >> quadric(versor << Point)

## 2. Shapes as sums of circle dyads

A ring is the band between two circles of the same centre, inside where a point is inside one and outside the other: the symmetric dyad of the two circles. A figure eight, Bernoulli's lemniscate, is the dyad of the circle through the pole less a weighted difference of two coordinate dyads. An hourglass is a Cassini oval of two tilted planes tangent at its two foci, pinched at the waist.

In [ ]:
def make_spherical_donut(r_core: float, r_tube: float) -> Quadric:
    """Spherical donut (torus) extensor on S², a map Plane <- Point: the band between the circles at the
    outer and inner edges of the ring, inside where a point lies inside one and outside the other."""
    outer = mv.z + np.cos(r_core + r_tube) * mv.w
    inner = mv.z + np.cos(r_core - r_tube) * mv.w
    return 0.5 * (outer * (inner & Point) + inner * (outer & Point))


def make_conical_donut(a: float, b: float) -> Quadric:
    """Spherical donut (Limaçon) with central hole meeting in a sharp conical apex on S²."""
    line = mv.w + mv.z + a * mv.x
    return line * (line & Point) - b**2 * (mv.x * (mv.x & Point) + mv.y * (mv.y & Point))


def make_bernoulli_lemniscate(scale_a: float) -> Quadric:
    """True Lemniscate of Bernoulli figure-8 on S²: (w - z)**2 - 2 * scale_a**2 * (x**2 - y**2) < 0, with the
    circle through the pole, `mv.w + mv.z`."""
    pole = mv.w + mv.z
    return pole * (pole & Point) - 2.0 * scale_a**2 * (mv.x * (mv.x & Point) - mv.y * (mv.y & Point))


def make_spherical_hourglass(theta: float, c_waist: float) -> Quadric:
    """True vertical hourglass on S²: two symmetric bulbs connected by a narrow waist."""
    # The plane z turned toward y and away from it by the same angle.
    p1 = mv.w + ((mv.zy * (-theta / 2)).exp() >> mv.z)
    p2 = mv.w + ((mv.zy * (theta / 2)).exp() >> mv.z)
    return 0.5 * (p1 * (p2 & Point) + p2 * (p1 & Point)) - c_waist * mv.w * (mv.w & Point)


conical_donut = moved(make_conical_donut(0.55, 0.22), (mv.xz * -0.20).exp())   # [] Plane <- Point
lemniscate = make_bernoulli_lemniscate(0.38)                                   # [] Plane <- Point
hourglass = make_spherical_hourglass(np.radians(24.0), 0.010)                  # [] Plane <- Point
quadrics = Extensor.stack([conical_donut, lemniscate, hourglass])              # [shapes] Plane <- Point
colors = ["#38bdf8", "#a855f7", "#f59e0b"]

In [ ]:
print("the lemniscate, a map from points to planes:\n", lemniscate.coefficient_table())

## 3. The flow

Two circles off the pole, each a tilted plane with its radius, meet in a 2-blade; its exponential at growing phases is a one-parameter family of conformal rotors, and each carries every shape at once. After a full turn of the phase the shapes are back where they started. The animation shows the front hemisphere, each shape in its colour where a point is inside it.

In [ ]:
# The two circles: each the plane z tilted toward x or y, and its radius; the frames of one turn, the
# pixels across, the traced pixels along each side of a shown one, and a frame's duration in milliseconds.
first_tilt, first_radius = 0.35, np.radians(48.0)
second_tilt, second_radius = 0.40, np.radians(52.0)
n_frames, resolution, frame_duration = 60, 300, 33

In [ ]:
def make_circle_intersection_vortex(
    center1: Plane, radius1: float, center2: Plane, radius2: float,
) -> Bivector:
    """Intersection 2-blade of two off-center circles on S²: each circle is the vector of its
    centre with weight the cosine of its radius."""
    c1 = center1 + mv.w * np.cos(radius1)
    c2 = center2 + mv.w * np.cos(radius2)
    return (c1 ^ c2).normalized()


def flow(quadrics: Quadric, generator: Bivector, phases: np.ndarray) -> Quadric:
    """The quadrics carried by (generator * (phase / 2)).exp(), batched as (phases, quadrics)."""
    rotors = (generator * (phases / 2.0)).exp()
    return moved(quadrics, rotors[:, None])


generator = make_circle_intersection_vortex((mv.zx * (-first_tilt / 2)).exp() >> mv.z, first_radius,
                                            (mv.zy * (-second_tilt / 2)).exp() >> mv.z, second_radius)   # [] Bivector
world = flow(quadrics, generator, np.linspace(0.0, 2.0 * np.pi, n_frames, endpoint=False))   # [frames, shapes] Plane <- Point
show_animation(render.vortex_frames(world, colors, resolution), "cga_trio_vortex", frame_duration)

In [ ]:
# checks
# The flow is conformal, so points of the sphere stay null; and a full turn brings every quadric back.
probes = point(np.array([[0.6, 0.0, 0.8], [0.0, -0.28, 0.96], [0.36, 0.48, 0.8]]))
carried = (generator * 0.4).exp() >> probes
np.testing.assert_allclose((carried | carried).to_array(), 0.0, atol=1e-12)
returned = flow(quadrics, generator, np.array([2.0 * np.pi]))[0]
np.testing.assert_allclose((returned(probes[:, None]) & probes[:, None]).to_array(),
                           (quadrics(probes[:, None]) & probes[:, None]).to_array(), atol=1e-6)